# Classification Model Using PyTorch

Follow-up to `linear_model.ipynb`: binary classification on `sklearn.datasets.make_circles` — a dataset deliberately not separable by a straight line. Goal: train a plain logistic regression first and watch it fail, then add one hidden layer with a `ReLU` activation and watch it succeed, making concrete *why* hidden layers and nonlinear activations exist rather than just introducing them as fact.

## Setup

Same `uv` project and environment as `linear_model.ipynb` — `torch`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn` are already installed, nothing new to add. Same `device` resolution pattern reused as-is.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme()

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"using device: {device}")

## Block 1 — Data: Circles

`sklearn.datasets.make_circles` generates two classes arranged as concentric rings — an inner circle (one class) surrounded by an outer ring (the other class). This shape is chosen deliberately: **no straight line can separate an inner circle from a ring around it.** Any single straight cut through the plane either misses part of the inner circle or includes part of the outer ring. This sets up Block 4a's planned failure — a plain logistic regression can only draw a straight decision boundary, so it's mathematically guaranteed to fail on this specific shape, not because of a bug or bad luck.

Same convention as `linear_model.ipynb`: `sklearn.model_selection.train_test_split` for the train/val split, `numpy` for array handling.

In [ ]:
from sklearn.datasets import make_circles

X, y = make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42)

print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"class counts: {np.bincount(y)}")

plt.figure(figsize=(5, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap="coolwarm", edgecolors="k", s=25)
plt.xlabel("x1")
plt.ylabel("x2")
plt.title("make_circles: two classes, not linearly separable")
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split

X_train_np, X_val_np, y_train_np, y_val_np = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# (N, 2) features, (N, 1) labels, both float32 for BCEWithLogitsLoss later
X_train = torch.tensor(X_train_np, dtype=torch.float32).to(device)
y_train = torch.tensor(y_train_np, dtype=torch.float32).reshape(-1, 1).to(device)
X_val = torch.tensor(X_val_np, dtype=torch.float32).to(device)
y_val = torch.tensor(y_val_np, dtype=torch.float32).reshape(-1, 1).to(device)

print(f"train: X={tuple(X_train.shape)}  y={tuple(y_train.shape)}")
print(f"val:   X={tuple(X_val.shape)}  y={tuple(y_val.shape)}")

`X shape: (500, 2)` — 500 points, 2 features each (this is a 2D dataset, unlike the linear notebook's 1 feature, since a 2D plane is what makes "decision boundary" a visual, drawable thing). `class counts: [250 250]` — perfectly balanced, `make_circles` splits samples evenly between the inner circle and outer ring by default. The scatter plot shows exactly what the rationale predicted: red and blue points arranged as two nested rings, visibly interleaved at the boundary — there's no way to draw one straight line through this plot that puts all of one color on one side.

After the split: 400 train / 100 val (80/20, `stratify=y` keeps the balance even in the split), `X` shaped `(N, 2)`, `y` reshaped to `(N, 1)` — same reasoning as the linear notebook's Block 3 reshape, `nn.Linear`/`BCEWithLogitsLoss` expect that shape, not a flat `(N,)`.

**Self-check passed:** shapes and class balance match expectations, scatter plot visibly shows two non-linearly-separable rings. Next up is Block 2 — a light recap of sigmoid and binary cross-entropy before reaching for `nn.Module`.

## Block 2 — Mechanics Recap (Light): Sigmoid & BCE

Two new pieces needed for classification that regression didn't use:

**Sigmoid** — `σ(z) = 1 / (1 + e^-z)` — squashes any real number `z` (a "logit") into `(0, 1)`, so it can be read as a probability. Large positive `z` → close to `1`; large negative `z` → close to `0`; `z = 0` → exactly `0.5`.

**Binary cross-entropy (BCE)** — `L = -[y·log(p) + (1-y)·log(1-p)]` — the loss used for classification instead of squared error. `y` is the true label (`0` or `1`), `p` is the predicted probability. Only one of the two terms is ever "active" per example (whichever matches the true label), and the loss blows up toward infinity the more confidently wrong a prediction is — squared error doesn't punish confident wrongness nearly as sharply, which is why it isn't used here.

The manual gradient-descent loop mechanic (predict → loss → `.backward()` → manual update → zero grad, repeated) was already fully proven in `linear_model.ipynb` Block 2 — no need to redo a full loop here. Just one scalar check to confirm autograd handles this new loss function correctly, same spirit as before.

In [ ]:
z = torch.linspace(-10, 10, 200)
p = torch.sigmoid(z)

plt.figure(figsize=(5, 3.5))
plt.plot(z.numpy(), p.numpy())
plt.axhline(0.5, color="gray", linestyle="--", linewidth=1)
plt.axvline(0, color="gray", linestyle="--", linewidth=1)
plt.xlabel("z (logit)")
plt.ylabel("σ(z)")
plt.title("Sigmoid")
plt.show()

In [ ]:
# scalar check: one point, true label y=1
w = torch.tensor(1.0, requires_grad=True)
x_point = torch.tensor(2.0)
y_point = torch.tensor(1.0)

z_point = w * x_point
p_point = torch.sigmoid(z_point)
loss = -(y_point * torch.log(p_point) + (1 - y_point) * torch.log(1 - p_point))
loss.backward()

# hand-computed derivative: dL/dw = (p - y) * x
hand_grad = (p_point.item() - y_point.item()) * x_point.item()

print(f"p = {p_point.item():.4f}, loss = {loss.item():.4f}")
print(f"w.grad (autograd) = {w.grad.item():.4f}")
print(f"hand-computed grad = {hand_grad:.4f}")

The sigmoid plot has exactly the expected S-shape: `σ(-10)≈0.00005` (essentially 0), `σ(0)=0.5` exactly, `σ(10)≈0.99995` (essentially 1) — confirmed by the curve flattening at both ends and crossing exactly through `(0, 0.5)`, marked by the dashed gray lines.

The scalar check: `w=1.0`, `x=2.0`, true label `y=1`, giving logit `z=2.0`, `p=σ(2.0)=0.8808`, loss `=0.1269`. **`w.grad` (autograd) = `-0.2384`, matching the hand-computed `dL/dw = (p-y)·x = (0.8808-1)·2 = -0.2384`** exactly — same verification pattern as the linear notebook, now confirmed for BCE + sigmoid instead of squared error. The negative sign makes sense: since `p=0.8808 < y=1`, the prediction is a little too low, so increasing `w` (and thus `z` and `p`) would reduce the loss — gradient descent would push `w` up, same interpretation as before.

**Self-check passed:** sigmoid plot matches the expected shape, autograd gradient matches the hand-computed derivative exactly. Block 3 moves to `nn.Module` — briefer than the linear notebook's version, since the manual-to-`nn` mapping concept was already taught there.

## Block 3 — `nn.Module` for Classification (Brief)

The `nn`-equivalent mapping concept was already taught in full in `linear_model.ipynb` Block 3 — the same idea applies here, just adapted: `nn.Linear(2, 1)` (2 in, since the circles data has 2 features; 1 out, a single logit) replaces manual `w`/`b`, and the loss changes from `nn.MSELoss()` to a BCE-based loss.

**Gotcha worth calling out explicitly:** PyTorch's `nn.BCEWithLogitsLoss()` expects **raw logits** as input — not post-sigmoid probabilities. It applies `sigmoid` internally, combined with the BCE formula, in one numerically stable step (computing `log(sigmoid(z))` directly avoids precision problems that computing `sigmoid(z)` then `log(...)` separately can hit for very confident/very wrong predictions). This means: **the model's final layer should NOT include a `Sigmoid()`** — just output the raw logit, and let `BCEWithLogitsLoss` handle the rest. This is different from `nn.BCELoss()`, which *does* expect probabilities already in `(0, 1)` (i.e. you'd need a `Sigmoid()` in the model first). Both Block 4a and 4b use `BCEWithLogitsLoss` with no `Sigmoid()` in the model.

In [ ]:
model = nn.Sequential(nn.Linear(2, 1)).to(device)
print(model)

loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

# one manual step on the real train batch
logits_before = model(X_train)
loss_before = loss_fn(logits_before, y_train)

optimizer.zero_grad()
loss_before.backward()
optimizer.step()

logits_after = model(X_train)
loss_after = loss_fn(logits_after, y_train)

print(f"loss before 1 step: {loss_before.item():.4f}")
print(f"loss after  1 step: {loss_after.item():.4f}")

`print(model)` confirms `in_features=2, out_features=1` — matches the circles data's 2 features and a single logit output, as expected. Notice there's no `Sigmoid()` anywhere in the model — `BCEWithLogitsLoss` needs the raw logit, per the gotcha above.

Loss goes from `0.7373` to `0.7354` after one `zero_grad → backward → step` cycle (this run — yours will differ, random init isn't seeded here, same as the linear notebook's Block 3 demo) — a small but real decrease, confirming the pattern works correctly on this new loss function and 2D input. (Loss starting near `~0.69` would be "pure random guessing" territory — `-log(0.5) ≈ 0.693` — so a value like `0.7373` just reflects this particular random init being a bit off from neutral, nothing concerning.)

**Self-check passed:** one step reduced the loss, model shape is correct. Block 4a trains this properly — full epochs, not just one step — and checks what a plain logistic regression can actually achieve on data it fundamentally cannot solve.

## Block 4a — Project Part 1: Logistic Regression FAILS

Train the same architecture as Block 3 properly this time — full epochs on the real train/val split, `model.train()`/`model.eval()` around the right passes, exactly mirroring the linear notebook's Block 4 training loop. The only real question here is: **how well can a single linear layer do on data that's fundamentally not linearly separable?**

This block also introduces the one genuinely new technique the classification notebook needs: plotting a **decision boundary**. The idea: build a fine grid of points covering the 2D feature space, ask the trained model to classify every single grid point, then color each grid cell by its predicted class (`matplotlib`'s `contourf`) and overlay the real data on top. Wherever the color changes is the model's actual decision boundary — this is reused as a plain function for Block 4b too, not redefined.

In [ ]:
torch.manual_seed(42)

model_logreg = nn.Sequential(nn.Linear(2, 1)).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_logreg.parameters(), lr=0.5)

n_epochs = 300
train_losses, val_losses = [], []

for epoch in range(n_epochs):
    model_logreg.train()
    logits_train = model_logreg(X_train)
    loss_train = loss_fn(logits_train, y_train)

    optimizer.zero_grad()
    loss_train.backward()
    optimizer.step()

    model_logreg.eval()
    with torch.no_grad():
        logits_val = model_logreg(X_val)
        loss_val = loss_fn(logits_val, y_val)

    train_losses.append(loss_train.item())
    val_losses.append(loss_val.item())

    if epoch % 50 == 0:
        print(
            f"epoch {epoch:4d}  train_loss={loss_train.item():.4f}  val_loss={loss_val.item():.4f}"
        )

print(f"\nfinal train_loss={train_losses[-1]:.4f}  val_loss={val_losses[-1]:.4f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(train_losses, label="train")
plt.plot(val_losses, label="val")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("Logistic regression: training vs validation loss")
plt.legend()
plt.show()

In [ ]:
def plot_decision_boundary(model, X_np, y_np, title):
    model.eval()
    x_min, x_max = X_np[:, 0].min() - 0.5, X_np[:, 0].max() + 0.5
    y_min, y_max = X_np[:, 1].min() - 0.5, X_np[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))
    grid = np.stack([xx.ravel(), yy.ravel()], axis=1).astype(np.float32)
    grid_t = torch.tensor(grid).to(device)

    with torch.no_grad():
        logits = model(grid_t)
        preds = (torch.sigmoid(logits) >= 0.5).float().cpu().numpy().reshape(xx.shape)

    plt.figure(figsize=(5, 5))
    plt.contourf(xx, yy, preds, alpha=0.3, cmap="coolwarm")
    plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="coolwarm", edgecolors="k", s=25)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.show()


plot_decision_boundary(
    model_logreg, X_val_np, y_val_np, "Logistic regression decision boundary"
)

val_preds = (torch.sigmoid(model_logreg(X_val)) >= 0.5).float()
val_accuracy = (val_preds == y_val).float().mean().item()
print(f"validation accuracy: {val_accuracy:.4f}")

**Confirmed: logistic regression genuinely fails here, and the numbers show exactly why.** Training loss plateaus almost immediately at `0.6928` and stays there — this is essentially `ln(2) ≈ 0.6931`, the loss value for *pure random guessing* (predicting `p=0.5` for everything gives BCE loss `-log(0.5) = ln(2)`). The loss curve isn't struggling to converge — it converges *instantly* to "no better than a coin flip" and then goes nowhere, because there's genuinely nowhere better to go.

The learned weights confirm it: `weight ≈ [0.0546, 0.0753]`, `bias ≈ -0.0004` — tiny numbers, barely different from the random init. The model searched for a useful straight-line direction and essentially found none, because none exists for this data. **Validation accuracy: `41%`** — at or below chance (50%), not an improvement over guessing.

The decision boundary plot shows exactly why: a single straight line slicing through the plane, putting roughly half the inner circle and half the outer ring on each side. No amount of additional training changes this — it's not a matter of more epochs or a better learning rate, it's a hard mathematical ceiling: **one linear layer can only ever produce a linear (straight-line) decision boundary**, and no straight line can separate a ring from the circle it surrounds.

**Self-check passed (as an expected failure):** accuracy ≈ chance level, boundary is visibly a straight line. Block 4b adds exactly one hidden layer with a nonlinear activation and revisits this same data.

## Block 4b — Project Part 2: Add a Hidden Layer + ReLU — It Works

A single `nn.Linear` layer can only bend the data through one straight cut, no matter how it's trained — that's the ceiling Block 4a hit. The fix: insert a **hidden layer** with a **nonlinear activation** (`nn.ReLU()`) between two linear layers. `ReLU(z) = max(0, z)` — zero for negative inputs, unchanged for positive ones. On its own that looks trivial, but stacking `Linear → ReLU → Linear` lets the network combine several straight "folds" of the input space into a single, genuinely curved decision boundary. This is the entire idea behind "deep" learning: linear layers alone stay linear no matter how many you stack (a line of a line is still a line), but a nonlinearity between them is what lets the composition curve.

Same training loop, same loss function, same data, same decision-boundary function as Block 4a — only the model architecture changes, from `nn.Sequential(nn.Linear(2, 1))` to `nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1))` (2 inputs → 8 hidden units → 1 output logit).

In [ ]:
torch.manual_seed(42)

model_mlp = nn.Sequential(
    nn.Linear(2, 8),
    nn.ReLU(),
    nn.Linear(8, 1),
).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_mlp.parameters(), lr=0.5)

n_epochs = 300
train_losses_mlp, val_losses_mlp = [], []

for epoch in range(n_epochs):
    model_mlp.train()
    logits_train = model_mlp(X_train)
    loss_train = loss_fn(logits_train, y_train)

    optimizer.zero_grad()
    loss_train.backward()
    optimizer.step()

    model_mlp.eval()
    with torch.no_grad():
        logits_val = model_mlp(X_val)
        loss_val = loss_fn(logits_val, y_val)

    train_losses_mlp.append(loss_train.item())
    val_losses_mlp.append(loss_val.item())

    if epoch % 50 == 0:
        print(
            f"epoch {epoch:4d}  train_loss={loss_train.item():.4f}  val_loss={loss_val.item():.4f}"
        )

print(
    f"\nfinal train_loss={train_losses_mlp[-1]:.4f}  val_loss={val_losses_mlp[-1]:.4f}"
)

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(train_losses_mlp, label="train")
plt.plot(val_losses_mlp, label="val")
plt.xlabel("epoch")
plt.ylabel("BCE loss")
plt.title("MLP (hidden layer + ReLU): training vs validation loss")
plt.legend()
plt.show()

In [ ]:
plot_decision_boundary(
    model_mlp, X_val_np, y_val_np, "MLP (hidden layer + ReLU) decision boundary"
)

val_preds_mlp = (torch.sigmoid(model_mlp(X_val)) >= 0.5).float()
val_accuracy_mlp = (val_preds_mlp == y_val).float().mean().item()
print(f"validation accuracy: {val_accuracy_mlp:.4f}")

**The payoff — a total reversal from Block 4a.** Loss actually decreases this time, steadily: `0.6884 → 0.5925 → 0.4261 → 0.2423 → 0.1407 → 0.0888`, ending at `train_loss=0.0614, val_loss=0.0777`. Compare to Block 4a's loss, which flatlined instantly at `0.6928` and never moved — this one keeps improving epoch after epoch because there's now an actual curved boundary for it to find.

**Validation accuracy: `1.0000`** — 100%, a complete reversal of Block 4a's `41%`. The decision boundary plot shows why: instead of one straight line, the boundary now curves into a ring shape, closely tracking the actual circular structure of the data — the inner circle's region and the outer ring's region are correctly separated almost everywhere.

The only thing that changed between Block 4a and 4b is `nn.Linear(2, 1)` → `nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1)` — same data, same loss function, same training loop, same learning rate. The one hidden layer with a nonlinear activation is entirely responsible for the jump from "can't do better than a coin flip" to "gets it exactly right." This is the concrete demonstration of why hidden layers and nonlinearities exist: they're not an optional refinement, they're what makes certain problems solvable at all for a neural network.

**Self-check passed:** accuracy ≥ ~90% target, easily exceeded at `100%`; boundary is visibly curved, not a straight line. Block 5 reviews production practices and saves this winning model.

## Block 5 — Production Practices (Lighter — Apply, Don't Re-Teach)

Same practices as `linear_model.ipynb` Block 5, already applied throughout 4a/4b without re-explaining them each time:

- **Reproducible seeding** — `torch.manual_seed(42)` before both `model_logreg` and `model_mlp` were built.
- **Device-agnostic code** — `.to(device)` on the model and every tensor, same `device` resolved once in Setup.
- **`model.train()` / `model.eval()`** — set correctly around the train and validation passes in both training loops.
- **No gradient tracking during validation/inference** — `torch.no_grad()` used for validation loss and for the entire decision-boundary grid pass.

One thing not shown yet: saving and reloading the winning model (`model_mlp`) so its trained weights survive beyond this notebook session.

In [ ]:
# save the winning MLP's parameters
torch.save(model_mlp.state_dict(), "classification_model_state.pt")
print("saved to classification_model_state.pt")

# build a FRESH model — different random init — then load the saved weights into it
fresh_model = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 1)).to(device)

fresh_model.load_state_dict(torch.load("classification_model_state.pt"))
fresh_model.eval()

with torch.no_grad():
    pred_original = torch.sigmoid(model_mlp(X_val))
    pred_reloaded = torch.sigmoid(fresh_model(X_val))

print("predictions match:", torch.allclose(pred_original, pred_reloaded))

**`predictions match: True`** — `fresh_model`, built with a completely different random init and never trained, produces *identical* predictions to the trained `model_mlp` once it loads the saved `state_dict()`. Same conclusion and same mechanism as `linear_model.ipynb` Block 5: `state_dict()` only stores the learned numbers (the `Linear` layers' weights/biases), not the architecture or any Python code — as long as a fresh model is built with the exact same architecture (`nn.Linear(2,8), nn.ReLU(), nn.Linear(8,1)`) before loading, the weights slot back in exactly.

**Self-check passed:** reloaded model's predictions match the original's exactly. Block 6 wraps up with a results table comparing the two models and what's deliberately deferred to a future notebook.

## Block 6 — Wrap-up

| model | final train loss | final val loss | val accuracy | decision boundary |
|---|---|---|---|---|
| logistic regression (4a) | 0.6928 | 0.6963 | 41.00% | straight line |
| MLP: Linear→ReLU→Linear (4b) | 0.0614 | 0.0777 | 100.00% | curved, tracks the rings |

**One-sentence takeaway:** a linear model can only draw a straight decision boundary — adding one hidden layer with a nonlinear activation (`ReLU`) let the network learn a curved boundary and solve a problem a linear model was mathematically guaranteed to fail at, taking validation accuracy from 41% (worse than a coin flip) to 100%.

**Next steps / deferred** — deliberately not covered in this notebook, candidates for a follow-up:
- multi-class classification (`nn.CrossEntropyLoss`, softmax over more than 2 classes)
- confusion matrix / precision / recall beyond raw accuracy (this notebook's data was perfectly balanced, so accuracy alone was a fair metric — real, imbalanced classification problems usually aren't)
- `Dataset`/`DataLoader` classes for batching (same gap as the linear notebook — full-batch training every epoch, fine at 400 rows, not at scale)
- deeper/wider MLPs and regularization (dropout, weight decay) — this notebook's 8-unit hidden layer was already enough to solve circles perfectly; harder datasets need more capacity and ways to keep it from overfitting
- learning rate schedulers
- CNNs, once image data is introduced